In [3]:
import pandas as pd
import os

# =============================
# 1. 参数设置 (请务必检查路径!)
# =============================
# 假设您的文件路径结构类似，只是将 NK 换成了 Neutrophil
# 请修改为您实际的 cNMF 结果文件路径
INPUT_FILE = "/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.5.cNMF_Neutrophils/Example_refbuilder_Neutrophils/3.ALL_cGEP_top_genes_Neutrophils_wide_filt.csv"
OUT_DIR = "/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.5.cNMF_Neutrophils/3.Neutrophils_GEP_Anno/"
OUT_FILENAME = "3.Neutrophil_cGEP7_Annotation_Results.csv" 

# 确保输出目录存在
if not os.path.exists(OUT_DIR):
    try:
        os.makedirs(OUT_DIR)
    except Exception as e:
        print(f"⚠️ 无法创建目录，请检查权限或路径: {e}")
        # 如果无法创建，输出到当前目录
        OUT_DIR = "."

OUT_PATH = os.path.join(OUT_DIR, OUT_FILENAME)

# =============================
# 2. 定义中性粒细胞参考集
# =============================
# 基于人类中性粒细胞 GEP 注释表 (Human Neutrophil GEP Annotation)
marker_dict = {
    # --- A. Lineage: 发育与颗粒生成 (Development) ---
    'Neu_Lineage_Prog': {'CD34', 'KIT', 'SOX4', 'NPM1', 'FLT3', 'GATA2', 'MEIS1', 'MKI67', 'TOP2A', 'CDK6', 'TYMS'},
    'Neu_Lineage_Granule1': {'ELANE', 'MPO', 'PRTN3', 'AZU1', 'CTSG', 'DEFA3', 'DEFA4', 'DEFA1', 'RNASE2', 'RNASE3', 'SERPINB10'},
    'Neu_Lineage_Granule2': {'LTF', 'LCN2', 'MMP8', 'CAMP', 'PGLYRP1', 'TCN1', 'FOLR3', 'SLPI', 'CRISP3', 'CEACAM8', 'CHI3L1'},
    'Neu_Lineage_Mature': {'FCGR3B', 'CXCR2', 'ALPL', 'AQP9', 'G0S2', 'CMTM2', 'VNN2', 'FPR1', 'FPR2', 'SELPLG', 'C5AR1'},

    # --- B. Functional: 功能与极化状态 (Functional States) ---
    'Neu_Func_IFN': {'ISG15', 'IFIT1', 'IFIT2', 'IFIT3', 'MX1', 'RSAD2', 'GBP1', 'CXCL10', 'STAT1', 'TNFSF10', 'OAS1'},
    'Neu_Func_Inf': {'IL1B', 'CXCL8', 'TNF', 'CCL3', 'CCL4', 'ICAM1', 'NFKBIA', 'IER3', 'CD83', 'SOD2', 'NAMPT'},
    'Neu_Func_MDSC': {'OLR1', 'CD274', 'STAT3', 'IL1RN', 'S100A12', 'FTH1', 'ANXA1', 'THBS1', 'CD84', 'S100A8', 'S100A9'},
    'Neu_Func_Angio': {'VEGFA', 'MMP9', 'SPP1', 'PROK2', 'LGALS3', 'PLAUR', 'HP', 'BNIP3L', 'ADM'},
    'Neu_Func_Meta': {'LDHA', 'HK2', 'ENO1', 'GAPDH', 'PKM', 'PGK1', 'TPI1', 'ALDOA', 'SLC2A1', 'HIF1A', 'BNIP3'},

    # --- C. Artifact: 技术噪音与应激 (Technical Artifacts) ---
    'Neu_Art_Stress': {'JUN', 'JUNB', 'JUND', 'FOS', 'FOSB', 'DUSP1', 'ZFP36', 'NR4A1', 'HSP90AA1', 'HSPA1A'},
    'Neu_Art_Mito': {'MT-CO1', 'MT-CO2', 'MT-ND1', 'MT-ND2', 'MT-ATP6', 'MT-CYB'},
    'Neu_Art_Ribo': {'RPL13', 'RPL10', 'RPS9', 'RPS27', 'RPLP1'},

    # --- D. Doublet: 双粘连污染 (Contamination) ---
    'Neu_Dbl_Mono': {'CD14', 'MS4A6A', 'LYZ', 'C1QA', 'C1QB', 'FCGR3A', 'MAFB', 'CSF1R', 'HLA-DRA'},
    'Neu_Dbl_Eos': {'CLC', 'IL5RA', 'CCR3', 'SIGLEC8', 'EPX', 'RNASE2', 'RNASE3'},
    'Neu_Dbl_RBC': {'HBA1', 'HBA2', 'HBB', 'HBD'}
}

print(f"✅ 成功加载 {len(marker_dict)} 个中性粒细胞参考类型。")

# =============================
# 3. 读取数据
# =============================
print(f"正在读取文件: {INPUT_FILE}")
try:
    df = pd.read_csv(INPUT_FILE)
except FileNotFoundError:
    print(f"❌ 错误: 找不到文件 {INPUT_FILE}")
    print("👉 请修改脚本顶部的 INPUT_FILE 路径。")
    exit()

# =============================
# 4. 注释函数 (核心逻辑不变)
# =============================
def annotate_cgep_spectra(gene_list):
    input_genes = set(gene_list)
    
    best_results = {
        "Spectra_Label": "Unknown",
        "Overlap_Count": 0,
        "Jaccard_Score": 0.0,
        "Key_Markers": ""
    }
    
    max_score = -1
    
    for ref_name, ref_genes in marker_dict.items():
        # 计算交集
        intersection = input_genes & ref_genes
        if not intersection:
            continue
            
        overlap_n = len(intersection)
        
        # Jaccard Index 计算
        jaccard = overlap_n / len(input_genes | ref_genes)
        score = jaccard
        
        if score > max_score:
            max_score = score
            best_results = {
                "Spectra_Label": ref_name,
                "Overlap_Count": overlap_n,
                "Jaccard_Score": round(score, 4),
                "Key_Markers": ",".join(list(intersection))
            }
            
    return best_results

# =============================
# 5. 执行注释循环
# =============================
results = []

# 遍历每一列 (假设列名是 cGEP 名称)
for cgep_col in df.columns:
    # 1. 获取当前 cGEP 的基因
    genes = (
        df[cgep_col]
        .dropna()
        .astype(str)
        .str.upper() # 确保大小写匹配
        .tolist()
    )
    
    if not genes:
        continue

    # 2. 运行注释
    anno = annotate_cgep_spectra(genes)
    
    # 3. 收集结果
    results.append({
        "cGEP_Name": cgep_col,
        "Spectra_Label": anno["Spectra_Label"],
        "Overlap_Count": anno["Overlap_Count"],
        "Jaccard_Score": anno["Jaccard_Score"],
        "Key_Markers": anno["Key_Markers"],
        "Top_5_Genes_Input": ",".join(genes[:5]) 
    })

# =============================
# 6. 输出结果
# =============================
out_df = pd.DataFrame(results)

if not out_df.empty:
    # 按照分数降序排列
    out_df = out_df.sort_values(by="Jaccard_Score", ascending=False)
    out_df.to_csv(OUT_PATH, index=False)

    print("-" * 30)
    print("✅ 中性粒细胞 cGEP 注释完成！")
    print(f"📄 结果已保存至: {OUT_PATH}")
    print("-" * 30)
    print("预览前 5 行结果:")
    print(out_df[["cGEP_Name", "Spectra_Label", "Overlap_Count", "Jaccard_Score"]].head())
else:
    print("⚠️ 未生成任何结果，请检查输入文件格式是否正确。")

✅ 成功加载 15 个中性粒细胞参考类型。
正在读取文件: /sibcb1/bioinformatics/yangyue/project/immunotherapy/7.5.cNMF_Neutrophils/Example_refbuilder_Neutrophils/3.ALL_cGEP_top_genes_Neutrophils_wide_filt.csv
------------------------------
✅ 中性粒细胞 cGEP 注释完成！
📄 结果已保存至: /sibcb1/bioinformatics/yangyue/project/immunotherapy/7.5.cNMF_Neutrophils/3.Neutrophils_GEP_Anno/3.Neutrophil_cGEP7_Annotation_Results.csv
------------------------------
预览前 5 行结果:
  cGEP_Name  Spectra_Label  Overlap_Count  Jaccard_Score
4     cGEP5   Neu_Func_IFN              8         0.2424
3     cGEP4   Neu_Func_Inf              6         0.1714
5     cGEP6   Neu_Dbl_Mono              4         0.1143
2     cGEP3  Neu_Func_MDSC              4         0.1081
1     cGEP2   Neu_Art_Ribo              3         0.0938


In [2]:
out_df

,cGEP_Name,Spectra_Label,Overlap_Count,Jaccard_Score,Key_Markers,Top_5_Genes_Input
4,cGEP5,Neu_Func_IFN,8,0.2424,"RSAD2,IFIT2,MX1,IFIT1,STAT1,GBP1,IFIT3,ISG15","IFIT1,RSAD2,IFIT3,ISG15,MX1"
3,cGEP4,Neu_Func_Inf,6,0.1714,"NFKBIA,CXCL8,CD83,NAMPT,IL1B,IER3","IER3,IL1B,NFKBIA,NLRP3,AC095055.1"
5,cGEP6,Neu_Dbl_Mono,4,0.1143,"C1QB,CSF1R,C1QA,FCGR3A","FCGR3A,RYR3,AIF1,C1QA,LST1"
2,cGEP3,Neu_Func_MDSC,4,0.1081,"S100A12,ANXA1,S100A9,S100A8","S100A9,S100A12,S100A8,VCAN,LYZ"
1,cGEP2,Neu_Art_Ribo,3,0.0938,"RPS27,RPL10,RPL13","CXCR5,LINC01641,GAS5,RPS8,RPS18"
6,cGEP7,Neu_Dbl_Mono,1,0.0263,HLA-DRA,"HLA-DQA1,FCER1A,HLA-DRA,HLA-DPA1,HLA-DQA2"
0,cGEP1,Unknown,0,0.0000,,"CD3G,IL32,LCK,CCL5,CD3E"


In [4]:
import pandas as pd
import os

# =============================
# 1. 参数设置 (请务必检查路径!)
# =============================
# 假设您的文件路径结构类似，只是将 NK 换成了 Neutrophil
# 请修改为您实际的 cNMF 结果文件路径
INPUT_FILE2 = "/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.5.cNMF_Neutrophils/Example_refbuilder_Neutrophils/ALL_cGEP_top_genes_Neutrophils_wide.csv"
OUT_DIR = "/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.5.cNMF_Neutrophils/3.Neutrophils_GEP_Anno/"
OUT_FILENAME2 = "3.Neutrophil_cGEP30_Annotation_Results.csv" 

# 确保输出目录存在
if not os.path.exists(OUT_DIR):
    try:
        os.makedirs(OUT_DIR)
    except Exception as e:
        print(f"⚠️ 无法创建目录，请检查权限或路径: {e}")
        # 如果无法创建，输出到当前目录
        OUT_DIR = "."

OUT_PATH2 = os.path.join(OUT_DIR, OUT_FILENAME2)

# =============================
# 2. 定义中性粒细胞参考集
# =============================
# 基于人类中性粒细胞 GEP 注释表 (Human Neutrophil GEP Annotation)
marker_dict = {
    # --- A. Lineage: 发育与颗粒生成 (Development) ---
    'Neu_Lineage_Prog': {'CD34', 'KIT', 'SOX4', 'NPM1', 'FLT3', 'GATA2', 'MEIS1', 'MKI67', 'TOP2A', 'CDK6', 'TYMS'},
    'Neu_Lineage_Granule1': {'ELANE', 'MPO', 'PRTN3', 'AZU1', 'CTSG', 'DEFA3', 'DEFA4', 'DEFA1', 'RNASE2', 'RNASE3', 'SERPINB10'},
    'Neu_Lineage_Granule2': {'LTF', 'LCN2', 'MMP8', 'CAMP', 'PGLYRP1', 'TCN1', 'FOLR3', 'SLPI', 'CRISP3', 'CEACAM8', 'CHI3L1'},
    'Neu_Lineage_Mature': {'FCGR3B', 'CXCR2', 'ALPL', 'AQP9', 'G0S2', 'CMTM2', 'VNN2', 'FPR1', 'FPR2', 'SELPLG', 'C5AR1'},

    # --- B. Functional: 功能与极化状态 (Functional States) ---
    'Neu_Func_IFN': {'ISG15', 'IFIT1', 'IFIT2', 'IFIT3', 'MX1', 'RSAD2', 'GBP1', 'CXCL10', 'STAT1', 'TNFSF10', 'OAS1'},
    'Neu_Func_Inf': {'IL1B', 'CXCL8', 'TNF', 'CCL3', 'CCL4', 'ICAM1', 'NFKBIA', 'IER3', 'CD83', 'SOD2', 'NAMPT'},
    'Neu_Func_MDSC': {'OLR1', 'CD274', 'STAT3', 'IL1RN', 'S100A12', 'FTH1', 'ANXA1', 'THBS1', 'CD84', 'S100A8', 'S100A9'},
    'Neu_Func_Angio': {'VEGFA', 'MMP9', 'SPP1', 'PROK2', 'LGALS3', 'PLAUR', 'HP', 'BNIP3L', 'ADM'},
    'Neu_Func_Meta': {'LDHA', 'HK2', 'ENO1', 'GAPDH', 'PKM', 'PGK1', 'TPI1', 'ALDOA', 'SLC2A1', 'HIF1A', 'BNIP3'},

    # --- C. Artifact: 技术噪音与应激 (Technical Artifacts) ---
    'Neu_Art_Stress': {'JUN', 'JUNB', 'JUND', 'FOS', 'FOSB', 'DUSP1', 'ZFP36', 'NR4A1', 'HSP90AA1', 'HSPA1A'},
    'Neu_Art_Mito': {'MT-CO1', 'MT-CO2', 'MT-ND1', 'MT-ND2', 'MT-ATP6', 'MT-CYB'},
    'Neu_Art_Ribo': {'RPL13', 'RPL10', 'RPS9', 'RPS27', 'RPLP1'},

    # --- D. Doublet: 双粘连污染 (Contamination) ---
    'Neu_Dbl_Mono': {'CD14', 'MS4A6A', 'LYZ', 'C1QA', 'C1QB', 'FCGR3A', 'MAFB', 'CSF1R', 'HLA-DRA'},
    'Neu_Dbl_Eos': {'CLC', 'IL5RA', 'CCR3', 'SIGLEC8', 'EPX', 'RNASE2', 'RNASE3'},
    'Neu_Dbl_RBC': {'HBA1', 'HBA2', 'HBB', 'HBD'}
}

print(f"✅ 成功加载 {len(marker_dict)} 个中性粒细胞参考类型。")

# =============================
# 3. 读取数据
# =============================
print(f"正在读取文件: {INPUT_FILE2}")
try:
    df2 = pd.read_csv(INPUT_FILE2)
except FileNotFoundError:
    print(f"❌ 错误: 找不到文件 {INPUT_FILE2}")
    print("👉 请修改脚本顶部的 INPUT_FILE2 路径。")
    exit()

# =============================
# 4. 注释函数 (核心逻辑不变)
# =============================
def annotate_cgep_spectra(gene_list):
    input_genes = set(gene_list)
    
    best_results = {
        "Spectra_Label": "Unknown",
        "Overlap_Count": 0,
        "Jaccard_Score": 0.0,
        "Key_Markers": ""
    }
    
    max_score = -1
    
    for ref_name, ref_genes in marker_dict.items():
        # 计算交集
        intersection = input_genes & ref_genes
        if not intersection:
            continue
            
        overlap_n = len(intersection)
        
        # Jaccard Index 计算
        jaccard = overlap_n / len(input_genes | ref_genes)
        score = jaccard
        
        if score > max_score:
            max_score = score
            best_results = {
                "Spectra_Label": ref_name,
                "Overlap_Count": overlap_n,
                "Jaccard_Score": round(score, 4),
                "Key_Markers": ",".join(list(intersection))
            }
            
    return best_results

# =============================
# 5. 执行注释循环
# =============================
results = []

# 遍历每一列 (假设列名是 cGEP 名称)
for cgep_col in df2.columns:
    # 1. 获取当前 cGEP 的基因
    genes = (
        df2[cgep_col]
        .dropna()
        .astype(str)
        .str.upper() # 确保大小写匹配
        .tolist()
    )
    
    if not genes:
        continue

    # 2. 运行注释
    anno = annotate_cgep_spectra(genes)
    
    # 3. 收集结果
    results.append({
        "cGEP_Name": cgep_col,
        "Spectra_Label": anno["Spectra_Label"],
        "Overlap_Count": anno["Overlap_Count"],
        "Jaccard_Score": anno["Jaccard_Score"],
        "Key_Markers": anno["Key_Markers"],
        "Top_5_Genes_Input": ",".join(genes[:5]) 
    })

# =============================
# 6. 输出结果
# =============================
out_df2 = pd.DataFrame(results)

if not out_df2.empty:
    # 按照分数降序排列
    out_df2 = out_df2.sort_values(by="Jaccard_Score", ascending=False)
    out_df2.to_csv(OUT_PATH2, index=False)

    print("-" * 30)
    print("✅ 中性粒细胞 cGEP 注释完成！")
    print(f"📄 结果已保存至: {OUT_PATH2}")
    print("-" * 30)
    print("预览前 5 行结果:")
    print(out_df2[["cGEP_Name", "Spectra_Label", "Overlap_Count", "Jaccard_Score"]].head())
else:
    print("⚠️ 未生成任何结果，请检查输入文件格式是否正确。")

✅ 成功加载 15 个中性粒细胞参考类型。
正在读取文件: /sibcb1/bioinformatics/yangyue/project/immunotherapy/7.5.cNMF_Neutrophils/Example_refbuilder_Neutrophils/ALL_cGEP_top_genes_Neutrophils_wide.csv
------------------------------
✅ 中性粒细胞 cGEP 注释完成！
📄 结果已保存至: /sibcb1/bioinformatics/yangyue/project/immunotherapy/7.5.cNMF_Neutrophils/3.Neutrophils_GEP_Anno/3.Neutrophil_cGEP30_Annotation_Results.csv
------------------------------
预览前 5 行结果:
   cGEP_Name         Spectra_Label  Overlap_Count  Jaccard_Score
4      cGEP5          Neu_Func_IFN              8         0.2424
17    cGEP18  Neu_Lineage_Granule2              7         0.2059
15    cGEP16          Neu_Art_Mito              6         0.2000
3      cGEP4          Neu_Func_Inf              6         0.1714
5      cGEP6          Neu_Dbl_Mono              4         0.1143


In [5]:
out_df2

,cGEP_Name,Spectra_Label,Overlap_Count,Jaccard_Score,Key_Markers,Top_5_Genes_Input
4,cGEP5,Neu_Func_IFN,8,0.2424,"RSAD2,IFIT2,MX1,IFIT1,STAT1,GBP1,IFIT3,ISG15","IFIT1,RSAD2,IFIT3,ISG15,MX1"
17,cGEP18,Neu_Lineage_Granule2,7,0.2059,"CEACAM8,CRISP3,MMP8,LCN2,TCN1,LTF,CAMP","LCN2,RETN,LTF,CD24,CEACAM8"
15,cGEP16,Neu_Art_Mito,6,0.2000,"MT-ATP6,MT-ND2,MT-CYB,MT-CO1,MT-CO2,MT-ND1","MT-ATP6,MT-CO1,MT-ND4,MT-CO3,MT-CYB"
3,cGEP4,Neu_Func_Inf,6,0.1714,"NFKBIA,CXCL8,CD83,NAMPT,IL1B,IER3","IER3,IL1B,NFKBIA,NLRP3,AC095055.1"
5,cGEP6,Neu_Dbl_Mono,4,0.1143,"C1QB,CSF1R,C1QA,FCGR3A","FCGR3A,RYR3,AIF1,C1QA,LST1"
26,cGEP27,Neu_Art_Stress,4,0.1111,"HSP90AA1,JUN,JUND,HSPA1A","HSPA1A,JUN,KLF2,DNAJA4,IER5"
2,cGEP3,Neu_Func_MDSC,4,0.1081,"S100A12,ANXA1,S100A9,S100A8","S100A9,S100A12,S100A8,VCAN,LYZ"
1,cGEP2,Neu_Art_Ribo,3,0.0938,"RPS27,RPL10,RPL13","CXCR5,LINC01641,GAS5,RPS8,RPS18"
24,cGEP25,Neu_Dbl_Mono,3,0.0833,"MS4A6A,C1QA,C1QB","C1QB,C1QC,C1QA,TMEM176B,MS4A6A"
13,cGEP14,Neu_Func_MDSC,3,0.0789,"S100A12,S100A9,S100A8","S100A9,S100A12,MMP9,S100A8,RFLNB"
